In [0]:
%pip install mlflow transformers torch accelerate sentencepiece protobuf --quiet
dbutils.library.restartPython()

In [0]:
import mlflow
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# Set Unity Catalog as the model registry
mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = "sarvamai/sarvam-30b"
REGISTERED_MODEL_NAME = "dev_agent.naval.sarvam_30b"

In [0]:
import os
os.environ["HF_TOKEN"] = ""

In [0]:
# Load tokenizer and model with efficient settings for a 30B model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# Create a text-generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
)

In [0]:
# Infer signature from sample input/output
sample_input = "What is artificial intelligence?"
sample_output = generator(sample_input, max_new_tokens=50)

signature = mlflow.models.infer_signature(
    model_input=sample_input,
    model_output=sample_output,
)

print("Signature:")
print(signature)

In [0]:
# Log and register the model in Unity Catalog
with mlflow.start_run(run_name="sarvam-30b-registration"):
    model_info = mlflow.transformers.log_model(
        transformers_model=generator,
        artifact_path="model",
        signature=signature,
        input_example=["What is artificial intelligence?"],
        registered_model_name=REGISTERED_MODEL_NAME,
        task="llm/v1/chat",
    )
    print(f"Model URI: {model_info.model_uri}")
    print(f"Registered as: {REGISTERED_MODEL_NAME}")